# GeneResearcher: Jupyter User Guide

This notebook demonstrates the public Python API in a standard Jupyter environment. It works with local JupyterLab or Notebook, Google Colab, and other notebook services that support Python 3.10–3.12.

You will learn how to:

1. Prepare a dedicated Python environment.
2. Install GeneResearcher and a SciSpaCy sentence-processing model.
3. Load credentials from environment variables, a local `.env` file, or Colab Secrets.
4. Run a single-gene and a multi-gene analysis.
5. Run the six pipeline stages separately and inspect each stage's in-memory data.

Each analysis makes real NCBI, embedding, and language-model requests. The examples request up to two records from PubMed and two from PMC per gene; merging can yield more than two unique papers. Run only the examples you need while exploring.

## 0. Create and select a clean Python environment

GeneResearcher supports Python 3.10–3.12. SciSpaCy currently requires NumPy 1.x, which can conflict with packages in shared data-science environments. A dedicated environment keeps those dependencies separate.

Create the environment in a terminal **before opening this notebook**, then start Jupyter from that environment. Select the `Python (GeneResearcher)` kernel for this notebook.

### macOS and Linux

```bash
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install jupyterlab ipykernel
python -m ipykernel install --user --name generesearcher --display-name "Python (GeneResearcher)"
jupyter lab
```

### Windows PowerShell

```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install jupyterlab ipykernel
python -m ipykernel install --user --name generesearcher --display-name "Python (GeneResearcher)"
jupyter lab
```

If Jupyter is already running, register the kernel with the commands above and select it from the notebook's **Kernel** menu. Hosted notebook users can start a fresh runtime and use the installation cells below. Restart that runtime after installation if NumPy or spaCy was replaced.

## 1. Check the active notebook kernel

This cell verifies that the selected kernel uses a supported Python version. If it fails, create or select a Python 3.10–3.12 environment and reopen the notebook with that kernel.

In [ ]:
import sys

print("Python:", sys.version.split()[0])
print("Kernel executable:", sys.executable)

if not (3, 10) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("GeneResearcher requires Python 3.10, 3.11, or 3.12.")

## 2. Install GeneResearcher

This installs the latest PyPI release into the active kernel. The `dashscope` extra is included because the example configuration uses DashScope embeddings. Run this in the dedicated environment selected above.

If the installation changes NumPy or spaCy in a running kernel, restart the kernel before continuing, then resume at the version-check cell.

In [ ]:
%pip install "generesearcher[dashscope]"

## 3. Check the installed package version

The output should show the version installed from PyPI. The notebook uses top-level `await`, which is supported by standard Jupyter kernels.

In [ ]:
import importlib.metadata as metadata

print("GeneResearcher:", metadata.version("generesearcher"))

## 4. Install the SciSpaCy sentence-processing model

The default model is `en_core_sci_lg` (about 620 MB installed). You can select `en_core_sci_sm`, `en_core_sci_md`, or `en_core_sci_scibert` instead. Change `MODEL_NAME` before running the installer if you prefer another supported model.

In [ ]:
import subprocess

MODEL_NAME = "en_core_sci_lg"
subprocess.run(
    [sys.executable, "-m", "generesearcher", "install-spacy-model", "--model", MODEL_NAME],
    check=True,
)

### Verify the model

This confirms that the selected model can be loaded by the active kernel.

In [ ]:
import spacy

nlp = spacy.load(MODEL_NAME)
print("SciSpaCy model loaded:", MODEL_NAME)
del nlp

## 5. Provide API credentials

Set credentials before running an analysis. GeneResearcher reads process environment variables and a `.env` file in the current working directory. Keep `.env` out of version control and never put real keys in this notebook.

This example uses DashScope for embeddings and an OpenAI-compatible endpoint for the language model. Set these environment variables in your terminal, secret store, or untracked `.env` file:

| Environment variable | Example purpose |
| --- | --- |
| `ENTREZ_EMAIL` | Email address sent with NCBI Entrez requests |
| `EMBEDDING_API_KEY` | DashScope embedding key |
| `LLM_API_KEY` | Language-model API key |
| `LLM_MODEL_NAME` | Language-model name |
| `LLM_BASE_URL` | OpenAI-compatible language-model endpoint |
| `EXCLUDE_PMIDS` | Optional comma-separated PMID list |

For a local `.env`, use the canonical variable names above. In Colab, the next cell can optionally load Secrets named `ENTREZ_EMAIL`, `EMBEDDING_API_KEY`, `LLM_API_KEY`, and `EXCLUDE_PMIDS` into the process environment.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=False)

# Optional Colab Secrets support; other Jupyter environments use environment variables or .env.
try:
    from google.colab import userdata
except ImportError:
    userdata = None

loaded_secret_names = []
if userdata is not None:
    secret_mapping = {
        "ENTREZ_EMAIL": "ENTREZ_EMAIL",
        "EMBEDDING_API_KEY": "EMBEDDING_API_KEY",
        "LLM_API_KEY": "LLM_API_KEY",
        "EXCLUDE_PMIDS": "EXCLUDE_PMIDS",
    }
    for secret_name, environment_name in secret_mapping.items():
        try:
            secret_value = userdata.get(secret_name)
        except userdata.SecretNotFoundError:
            continue
        except userdata.NotebookAccessError as exc:
            raise RuntimeError(
                f"Grant this notebook access to the '{secret_name}' secret in Colab."
            ) from exc
        if secret_value:
            os.environ[environment_name] = secret_value
            loaded_secret_names.append(secret_name)

if loaded_secret_names:
    print("Loaded Colab Secrets:", ", ".join(loaded_secret_names))
else:
    print("Using the active process environment and any .env file found in the working directory.")

### Set example provider defaults and check required settings

These defaults match the example providers. Existing environment values take precedence, so you can use another compatible embedding or language-model service.

In [ ]:
os.environ.setdefault("EMBEDDING_PROVIDER", "dashscope")
os.environ.setdefault("EMBEDDING_MODEL_NAME", "text-embedding-v3")
# Set these to the model and OpenAI-compatible endpoint you use.
# os.environ["LLM_MODEL_NAME"] = "your-model-name"
# os.environ["LLM_BASE_URL"] = "https://your-llm-endpoint/v1"

required_settings = ["ENTREZ_EMAIL", "EMBEDDING_API_KEY", "LLM_API_KEY", "LLM_MODEL_NAME", "LLM_BASE_URL"]
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise RuntimeError(
        "Missing required settings: " + ", ".join(missing_settings)
        + ". Configure them in the process environment, .env, or your notebook secret store."
    )

excluded_pmids = [value.strip() for value in os.getenv("EXCLUDE_PMIDS", "").split(",") if value.strip()]
print("Required settings are available. Secret values were not displayed.")
print("PMID exclusions configured:", len(excluded_pmids))

## 6. Define a small example study

One `ResearchConfig` represents one target function and research background, with one or more genes. These examples request up to two search records from each of PubMed and PMC per gene. The merged, deduplicated total may exceed two. Increase the per-source limit for a real analysis.

In [ ]:
from generesearcher import AnalysisSession, ResearchConfig, run_analysis_async
from IPython.display import Markdown, display

FUNCTION_NAME = "histone acetylation"
BACKGROUND_TEMPLATE = (
    "What is the role of {gene} in {function_name}, especially in cancer development?"
)
KEYWORDS = ["cancer", "tumor"]
PAPERS_PER_GENE = 2

print("Function:", FUNCTION_NAME)
print("Per-source search limit:", PAPERS_PER_GENE)

## 7. Single-gene example: run the complete analysis

This runs all six stages for one target gene. The result is returned in memory; no project folder is required. Keep the result object while you inspect its data or report.

In [ ]:
single_gene_config = ResearchConfig(
    genes=["EP300"],
    function_name=FUNCTION_NAME,
    research_background_template=BACKGROUND_TEMPLATE,
    keywords=KEYWORDS,
    max_papers_per_gene=PAPERS_PER_GENE,
)

single_gene_result = await run_analysis_async(single_gene_config)
print("Analysis directory:", single_gene_result.analysis_directory)
print("Report exists:", single_gene_result.report_path.exists())

### Inspect the single-gene result

`AnalysisResult.analysis_data` contains the in-memory tables produced by the pipeline.

In [ ]:
single_gene_data = single_gene_result.analysis_data
print("Retrieved article records:", len(single_gene_data.raw_papers_df))
display(single_gene_data.raw_papers_df.head(5))
print("Synthesis records:", len(single_gene_data.synthesis_df))
display(single_gene_data.synthesis_df.head(5))
print("Report path:", single_gene_result.report_path)

## 8. Multi-gene example: run the complete analysis

This runs the same target function and research background for two genes. GeneResearcher performs gene-specific evidence analysis and adds the study-level synthesis used for a multi-gene study.

In [ ]:
multi_gene_config = ResearchConfig(
    genes=["EP300", "CREBBP"],
    function_name=FUNCTION_NAME,
    research_background_template=BACKGROUND_TEMPLATE,
    keywords=KEYWORDS,
    max_papers_per_gene=PAPERS_PER_GENE,
)

multi_gene_result = await run_analysis_async(multi_gene_config)
print("Analysis directory:", multi_gene_result.analysis_directory)
print("Report exists:", multi_gene_result.report_path.exists())

### Inspect the multi-gene result

The `Gene` column lets you check which records belong to each target gene.

In [ ]:
multi_gene_data = multi_gene_result.analysis_data
print("Retrieved article records:", len(multi_gene_data.raw_papers_df))
display(multi_gene_data.raw_papers_df.head(10))
print("Synthesis records by gene and relationship:")
display(multi_gene_data.synthesis_df[["Gene", "Analysis_Type", "Summary_Level", "PMID"]].head(20))
print("Report path:", multi_gene_result.report_path)

## 9. Step-by-step example: inspect every stage

`AnalysisSession` keeps the same analysis in the current Python process. Run each stage in order, one cell at a time, and inspect `session.analysis_data` after it completes. This example uses one gene; the six stages are the same stages used by the one-call API.

In [ ]:
stepwise_config = ResearchConfig(
    genes=["EP300"],
    function_name=FUNCTION_NAME,
    research_background_template=BACKGROUND_TEMPLATE,
    keywords=KEYWORDS,
    max_papers_per_gene=PAPERS_PER_GENE,
)

session = AnalysisSession(stepwise_config)
print("Next stage:", session.next_step)

### Step 1 — Retrieve literature and article text

The result table contains the article records retrieved for the target gene.

In [ ]:
await session.search_literature()
print("Completed:", session.completed_steps)
print("Retrieved article records:", len(session.analysis_data.raw_papers_df))
display(session.analysis_data.raw_papers_df.head(10))

### Step 2 — Embed article text and retrieve candidate sentences

The recalled-sentence table shows the candidate evidence selected for the research question.

In [ ]:
await session.retrieve_candidate_sentences()
print("Completed:", session.completed_steps)
print("Recalled sentence records:", len(session.analysis_data.recalled_sentences_df))
display(session.analysis_data.recalled_sentences_df.head(10))

### Step 3 — Assess candidate evidence for relevance

Inspect the model's evidence decisions and processing status. An empty result should be interpreted together with the stage logs.

In [ ]:
await session.assess_candidate_evidence()
print("Completed:", session.completed_steps)
print("Relevance records:", len(session.analysis_data.relevance_filtered_df))
display(session.analysis_data.relevance_filtered_df.head(10))

### Step 4 — Synthesize evidence

This stage creates article-level and cross-article synthesis. If no literature passes Step 3, Step 4 stops with a clear error and Steps 5–6 should not be run.

In [ ]:
await session.synthesize_evidence()
print("Completed:", session.completed_steps)
print("Synthesis records:", len(session.analysis_data.synthesis_df))
display(
    session.analysis_data.synthesis_df[
        ["Gene", "Analysis_Type", "Summary_Level", "PMID", "Summary_Text"]
    ].head(10)
)

### Step 5 — Build gene-level and multi-gene reporting results

This stage prepares the overview and other report components. The `reporting_data` table remains available for inspection.

In [ ]:
await session.analyze_functions()
print("Completed:", session.completed_steps)
print("Reporting records:", len(session.analysis_data.reporting_data))
display(session.analysis_data.reporting_data.head(10))

### Step 6 — Generate the Markdown report

After this stage, transfer the completed session into an `AnalysisResult` so its in-memory data and temporary files have a clear owner.

In [ ]:
await session.generate_report()
stepwise_result = session.to_result()
print("Completed stages:", session.completed_steps)
print("Report path:", stepwise_result.report_path)
print("Report exists:", stepwise_result.report_path.exists())

### Preview the report

This displays the first part of the report. Open the path printed above to inspect the complete file.

In [ ]:
report_text = stepwise_result.report_path.read_text(encoding="utf-8")
display(Markdown(report_text[:4000]))

## 10. Optional: keep a copy of an analysis

By default, the API uses temporary storage. `save_analysis()` copies a completed result to a persistent project folder. Choose a new destination for each save.

In [ ]:
from datetime import datetime
from pathlib import Path
from generesearcher import save_analysis

save_directory = Path("generesearcher_saved_runs") / datetime.now().strftime("run_%Y%m%d_%H%M%S")
saved_analysis_directory = save_analysis(stepwise_result, save_directory)
print("Saved analysis:", saved_analysis_directory)

## 11. Optional practice

Replace the example genes with genes from your own study, keep the per-source search limit low, and compare the number of retrieved articles and synthesis records. Each run makes new external API requests.

In [ ]:
custom_config = ResearchConfig(
    genes=["BRCA1", "TP53"],  # Replace with genes from your study.
    function_name=FUNCTION_NAME,
    research_background_template=BACKGROUND_TEMPLATE,
    keywords=KEYWORDS,
    max_papers_per_gene=1,
)

# Uncomment to run the additional analysis:
# custom_result = await run_analysis_async(custom_config)

## 12. Clean up temporary results when finished

Run this cell after you have inspected or saved the outputs. It removes temporary analysis files owned by the result objects; a copy created with `save_analysis()` remains in its saved project folder.

In [ ]:
for result_name in ("single_gene_result", "multi_gene_result", "stepwise_result"):
    result = globals().get(result_name)
    if result is not None:
        result.cleanup()

if "session" in globals():
    session.close()

print("Temporary analysis files cleaned up.")